# 02 · Error analysis

Concrete examples behind the numbers in section 5 of `00_main.ipynb`: the question, the top 5 chunks from the chosen retriever with the judge's label, and what each model chose. The examples are drawn at random with a fixed seed, so they are not hand-picked and the notebook shows the same ones every time; change `n` or `seed` below to see others.

**Reads:** `results/raw/qa/`, `results/raw/runs/`, `results/raw/judge/`, `data/processed/` and `results/tables/retriever_choice.csv` (written by `00_main.ipynb`)

Groups:
1. Fixed: Gemma wrong without chunks, right with them, and a relevant chunk in the top 5
2. Broken: Gemma right without chunks, wrong with them, and no relevant chunk in the top 5
3. Wrong although a relevant chunk was in the top 5
4. Low letter mass: Gemma with chunks put less than half its probability on the option letters

This is material for the Discussion section; nothing here changes the results in `00_main.ipynb`.

In [ ]:
# Picks up changes saved in src/ without restarting the notebook
%load_ext autoreload
%autoreload 2

import textwrap

import numpy as np
import pandas as pd

from norrag.config import get_path, load_config
from norrag.metrics import letter_mass, retrieval_scores
from norrag.records import read_jsonl_df

config = load_config()
k = config["qa"]["k"]
n = 2  # examples per group
seed = config["seed"]
raw = get_path(config, "results_raw")

# The retriever chosen in section 2 of 00_main.ipynb
chosen = pd.read_csv(get_path(config, "tables") / "retriever_choice.csv", index_col=0).loc["chosen", "value"]
retriever_names = {"bm25": "BM25", "dense": "sentence-BERT"}  # the same names as in 00_main.ipynb
gemma_rag = f"Gemma + {retriever_names[chosen]}"

questions = pd.read_parquet(get_path(config, "data_processed") / "questions.parquet").set_index("qid")
run = read_jsonl_df(raw / "runs" / f"{chosen}.jsonl")
judge = read_jsonl_df(raw / "judge" / f"{config['judge']['model']}.jsonl").rename(columns={"label": "relevant"})
qa = {
    "Borealis": read_jsonl_df(raw / "qa" / "borealis-12b__no_rag.jsonl").set_index("qid"),
    "Gemma": read_jsonl_df(raw / "qa" / "gemma-12b__no_rag.jsonl").set_index("qid"),
    gemma_rag: read_jsonl_df(raw / "qa" / f"gemma-12b__rag_{chosen}.jsonl").set_index("qid"),
}

# The texts and judge labels of every chunk in a top k
top_k = run.set_index("qid").chunk_ids.str[:k]
needed = sorted({chunk_id for ids in top_k for chunk_id in ids})
chunk_texts = pd.read_parquet(get_path(config, "data_processed") / "chunks.parquet", columns=["chunk_id", "text"],
                              filters=[("chunk_id", "in", needed)]).set_index("chunk_id")["text"]
judge_label = judge.set_index(["qid", "chunk_id"]).relevant

In [ ]:
# One row per question: right or wrong for each condition, retrieval hit, and Gemma's letter mass with chunks
by_question = pd.DataFrame({name: results.correct for name, results in qa.items()})
by_question["hit"] = retrieval_scores(run, judge, k).set_index("qid").hit == 1
by_question["letter_mass"] = qa[gemma_rag].logprobs.apply(letter_mass)

groups = {
    "fixed": by_question[~by_question["Gemma"] & by_question[gemma_rag] & by_question.hit],
    "broken": by_question[by_question["Gemma"] & ~by_question[gemma_rag] & ~by_question.hit],
    "wrong despite a relevant chunk": by_question[~by_question[gemma_rag] & by_question.hit],
    "low letter mass": by_question[by_question.letter_mass < 0.5],
}
pd.Series({name: len(group) for name, group in groups.items()}, name="questions")

In [ ]:
def show(qid):
    # Print one question: the options (✓ = correct), what each model chose, and the top k chunks with the judge's label
    question = questions.loc[qid]
    print("=" * 100)
    print(qid)
    print(textwrap.fill(question.question, 100))
    for label, option in zip(question.labels, question.options):
        print(f"  {'✓' if label == question.answer else ' '} {label}: {option}")
    print()
    for name, results in qa.items():
        result = results.loc[qid]
        probabilities = ", ".join(f"{label} {p:.0%}" for label, p in zip(question.labels, np.exp(result.logprobs)))
        print(f"  {name:13} chose {result.predicted} ({'right' if result.correct else 'wrong'})   "
              f"letters: {probabilities}   letter mass {letter_mass(result.logprobs):.0%}")
    print(f"\n  Top {k} chunks from {retriever_names[chosen]} (judge: Y = relevant, N = not):")
    for rank, chunk_id in enumerate(top_k[qid], start=1):
        mark = "Y" if judge_label[(qid, chunk_id)] == 1 else "N"
        print(textwrap.fill(chunk_texts[chunk_id], 100, initial_indent=f"  [{rank}] {mark}  ", subsequent_indent=" " * 9))
    print()

## 1 · Fixed by retrieval

Gemma got these wrong without chunks and right with them, and a relevant chunk was in the top 5.

In [ ]:
for qid in groups["fixed"].sample(n, random_state=seed).index:
    show(qid)

## 2 · Broken by retrieval

Gemma got these right without chunks and wrong with them, and no chunk in the top 5 was relevant.

In [ ]:
for qid in groups["broken"].sample(n, random_state=seed).index:
    show(qid)

## 3 · Wrong although a relevant chunk was found

A chunk in the top 5 gave the answer, but Gemma still chose a wrong option.

In [ ]:
for qid in groups["wrong despite a relevant chunk"].sample(n, random_state=seed).index:
    show(qid)

## 4 · Low letter mass

Gemma with chunks put less than half its probability on the option letters: it would rather have started its answer with something else, so the chosen letter is close to a guess.

In [ ]:
for qid in groups["low letter mass"].sample(n, random_state=seed).index:
    show(qid)